In [1]:
# 1. 라이브러리 불러오기 및 실습 데이터 생성
# 현실적인 실습을 위해 일부러 비어있는 값(None, np.nan)을 포함한 데이터프레임을 생성합니다.

import pandas as pd
import numpy as np

# 실습용 샘플 데이터 생성
data = {
    '이름': ['송강', '김유정', '이도현', '고윤정', '황민현'],
    '나이': [28, np.nan, 27, 26, np.nan],
    '부서': ['영업부', '기획부', None, '개발부', '기획부'],
    '보너스': [500, 300, np.nan, np.nan, 400]
}
df = pd.DataFrame(data)
print("--- 원본 데이터프레임 ---")
print(df)

--- 원본 데이터프레임 ---
    이름    나이   부서    보너스
0   송강  28.0  영업부  500.0
1  김유정   NaN  기획부  300.0
2  이도현  27.0  NaN    NaN
3  고윤정  26.0  개발부    NaN
4  황민현   NaN  기획부  400.0


In [2]:
# 2. 결측값 조회하기 (Detection)
# 데이터를 받으면 가장 먼저 어디에 결측값이 얼마나 있는지 확인해야 합니다.

print("1. 각 셀별 결측값 여부 (isnull 또는 isna):")
# 결측치면 True, 정상 데이터면 False를 반환합니다.
print(df.isnull())
print("-" * 45)

print("2. [필수] 열별 결측값 개수 총합 확인:")
# True를 1로 계산하여 열마다 몇 개의 결측치가 있는지 보여줍니다. 가장 자주 씁니다.
print(df.isnull().sum())
print("-" * 45)

print("3. 결측치가 하나라도 있는 행(Row)만 골라보기:")
# isnull().any(axis=1)을 조건으로 주면 결측치가 포함된 행만 필터링합니다.
print(df[df.isnull().any(axis=1)])

1. 각 셀별 결측값 여부 (isnull 또는 isna):
      이름     나이     부서    보너스
0  False  False  False  False
1  False   True  False  False
2  False  False   True   True
3  False  False  False   True
4  False   True  False  False
---------------------------------------------
2. [필수] 열별 결측값 개수 총합 확인:
이름     0
나이     2
부서     1
보너스    2
dtype: int64
---------------------------------------------
3. 결측치가 하나라도 있는 행(Row)만 골라보기:
    이름    나이   부서    보너스
1  김유정   NaN  기획부  300.0
2  이도현  27.0  NaN    NaN
3  고윤정  26.0  개발부    NaN
4  황민현   NaN  기획부  400.0


In [3]:
# 3. 결측값 삭제하기 (Drop)
# 결측치가 너무 많거나 대체할 방법이 없을 때는 dropna()를 사용하여 삭제합니다.

# [주의] 아래 예제들은 원본을 지키기 위해 inplace=True를 하지 않았습니다.

# 방법 A: 결측치가 하나라도 포함된 행은 무조건 삭제
df_drop_any = df.dropna()
print("1. 결측치가 있는 행 제거:")
print(df_drop_any)
print("-" * 45)

# 방법 B: 특정 열에 결측치가 있을 때만 행 삭제
# 예: '부서'가 누락된 직원은 데이터로서 가치가 없다고 판단될 때
df_drop_dept = df.dropna(subset=['부서'])
print("2. '부서' 열에 결측치가 있는 행만 제거:")
print(df_drop_dept)

1. 결측치가 있는 행 제거:
   이름    나이   부서    보너스
0  송강  28.0  영업부  500.0
---------------------------------------------
2. '부서' 열에 결측치가 있는 행만 제거:
    이름    나이   부서    보너스
0   송강  28.0  영업부  500.0
1  김유정   NaN  기획부  300.0
3  고윤정  26.0  개발부    NaN
4  황민현   NaN  기획부  400.0


In [5]:
# 4. 결측값 채우기 (Imputation)
# 데이터를 무작정 지우면 손실이 크므로, fillna()를 사용해 적절한 값으로 채워 넣는 것이 일반적입니다.

# 방법 A: 특정 상숫값으로 채우기
# 보너스에 누락된 값(NaN)은 보너스를 받지 못한 것이므로 0으로 채웁니다.
df_filled_zero = df.copy()
df_filled_zero['보너스'] = df_filled_zero['보너스'].fillna(0)
print("1. 보너스의 누락값을 0으로 대체:")
print(df_filled_zero)
print("-" * 45)

# 방법 B: 대표값(평균값, 중앙값 등)으로 채우기
# 나이의 누락값을 나머지 인원의 '평균 나이'로 계산해서 채웁니다.
df_filled_mean = df.copy()
mean_age = df_filled_mean['나이'].mean() # 평균 계산 (27.0세)

df_filled_mean['나이'] = df_filled_mean['나이'].fillna(mean_age)
print(f"2. 나이의 누락값을 평균 나이({mean_age}세)로 대체:")
print(df_filled_mean)
print("-" * 45)

# 방법 C: 문자열 데이터(범주형) 채우기
# 부서가 누락된 경우 '미정' 혹은 가장 빈도가 높은 '최빈값'으로 채울 수 있습니다.
df_filled_str = df.copy()
df_filled_str['부서'] = df_filled_str['부서'].fillna('대기발령')
print("3. 부서 누락값을 특정 문자열로 대체:")
print(df_filled_str)

# 방법 D: 앞/뒤 데이터로 채우기 (시계열 데이터용)
# 시간 순서대로 배열된 데이터나 흐름이 있는 데이터에서 주로 사용합니다.

# ffill: 결측치 바로 위의 행(앞의 데이터) 값으로 채움
# bfill: 결측치 바로 아래의 행(뒤의 데이터) 값으로 채움
df_time_series = df.copy()
print("4. 바로 직전 행의 값으로 채우기(ffill):")
print(df_time_series['부서'].ffill())

1. 보너스의 누락값을 0으로 대체:
    이름    나이   부서    보너스
0   송강  28.0  영업부  500.0
1  김유정   NaN  기획부  300.0
2  이도현  27.0  NaN    0.0
3  고윤정  26.0  개발부    0.0
4  황민현   NaN  기획부  400.0
---------------------------------------------
2. 나이의 누락값을 평균 나이(27.0세)로 대체:
    이름    나이   부서    보너스
0   송강  28.0  영업부  500.0
1  김유정  27.0  기획부  300.0
2  이도현  27.0  NaN    NaN
3  고윤정  26.0  개발부    NaN
4  황민현  27.0  기획부  400.0
---------------------------------------------
3. 부서 누락값을 특정 문자열로 대체:
    이름    나이    부서    보너스
0   송강  28.0   영업부  500.0
1  김유정   NaN   기획부  300.0
2  이도현  27.0  대기발령    NaN
3  고윤정  26.0   개발부    NaN
4  황민현   NaN   기획부  400.0
4. 바로 직전 행의 값으로 채우기(ffill):
0    영업부
1    기획부
2    기획부
3    개발부
4    기획부
Name: 부서, dtype: str
